# Step 1: Load and Inspect Your Datasets

In [ ]:
import pandas as pd

# Load the datasets (replace with your actual file paths)
try:
    df_sarcasm = pd.read_csv('/content/drive/MyDrive/NLP prj/sarcasm/train-balanced-sarcasm.csv')
    df_jokes = pd.read_csv('/content/drive/MyDrive/NLP prj/humor/shortjokes.csv')
except FileNotFoundError:
    print("Make sure your CSV files are in the correct folders!")
    exit()


# Inspect the sarcasm dataset
print("--- Sarcasm Dataset Info ---")
print(df_sarcasm.head())
print(df_sarcasm.info())
print("\nSarcasm label counts:")
print(df_sarcasm['label'].value_counts())

# Inspect the humor dataset
print("\n--- Humor Dataset Info ---")
print(df_jokes.head())
print(df_jokes.info())

--- Sarcasm Dataset Info ---
   label                                            comment     author  \
0      0                                         NC and NH.  Trumpbart   
1      0  You do know west teams play against west teams...  Shbshb906   
2      0  They were underdogs earlier today, but since G...   Creepeth   
3      0  This meme isn't funny none of the "new york ni...  icebrotha   
4      0                    I could use one of those tools.  cush2push   

            subreddit  score  ups  downs     date          created_utc  \
0            politics      2   -1     -1  2016-10  2016-10-16 23:55:23   
1                 nba     -4   -1     -1  2016-11  2016-11-01 00:24:10   
2                 nfl      3    3      0  2016-09  2016-09-22 21:45:37   
3  BlackPeopleTwitter     -8   -1     -1  2016-10  2016-10-18 21:03:47   
4  MaddenUltimateTeam      6   -1     -1  2016-12  2016-12-30 17:00:13   

                                      parent_comment  
0  Yeah, I get that argume

## Data preprocessing




In [ ]:
df_sarcasm.dropna(subset=['comment'], inplace=True)
df_jokes.dropna(subset=['Joke'], inplace=True)

df_sarcasm_processed = df_sarcasm[['comment', 'label']].copy()
df_jokes_processed = df_jokes[['Joke']].copy()

df_jokes_processed.rename(columns={'Joke': 'comment'}, inplace=True)
df_jokes_processed['label'] = 2

## Feature extraction




In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine the comment columns
combined_comments = pd.concat([df_sarcasm_processed['comment'], df_jokes_processed['comment']], ignore_index=True)

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the combined comments
tfidf_features = tfidf_vectorizer.fit_transform(combined_comments)

# Inspect the shape of the resulting feature matrix
print("Shape of TF-IDF feature matrix:", tfidf_features.shape)

# Optionally inspect the vocabulary (first 100 words)
# print("\nVocabulary (first 100 words):", list(tfidf_vectorizer.vocabulary_.keys())[:100])

Shape of TF-IDF feature matrix: (1242428, 188382)


## Combine and split data



In [ ]:
from sklearn.model_selection import train_test_split

# Combine the labels
combined_labels = pd.concat([df_sarcasm_processed['label'], df_jokes_processed['label']], ignore_index=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_features, combined_labels, test_size=0.2, random_state=42)

# Verify the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (993942, 188382)
Shape of X_test: (248486, 188382)
Shape of y_train: (993942,)
Shape of y_test: (248486,)


## Model selection




In [ ]:
# Model Choice Reasoning

# 1. Logistic Regression:
# Advantages: Relatively simple, interpretable, and computationally efficient. Good baseline model for text classification.
# Disadvantages: May not capture complex non-linear relationships in the data as well as more complex models.

# 2. Simple Neural Network (e.g., with a few dense layers):
# Advantages: Can learn more complex patterns in the data than linear models. Can be more powerful than Logistic Regression for larger datasets.
# Disadvantages: Can be more computationally expensive to train. Requires careful tuning of hyperparameters.

# Preliminary Decision:
# Given the large size of the dataset and the potential for complex patterns in sarcasm and humor,
# a simple Neural Network is a good preliminary choice. It offers a balance between complexity and
# computational cost compared to very deep models like LSTMs or Transformers, while likely
# outperforming a simple Logistic Regression on this task. We can start with a basic neural network
# and consider more complex architectures later if needed.

## Model training

### Subtask:
Train the selected model (Simple Neural Network) on the training data (`X_train`, `y_train`).


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the model architecture
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')  # 3 classes: 0 (not sarcastic), 1 (sarcastic), 2 (humor)
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=1,  # You can adjust the number of epochs
                    batch_size=512, # You can adjust the batch size
                    validation_split=0.2) # Use 20% of training data for validation

# Display model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1554/1554 ━━━━━━━━━━━━━━━━━━━━ 510s 326ms/step - accuracy: 0.6337 - loss: 0.7518 - val_accuracy: 0.6953 - val_loss: 0.6423


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │    24,113,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72,364,427 (276.05 MB)

 Trainable params: 24,121,475 (92.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 48,242,952 (184.03 MB)

## Model evaluation




In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test set
y_pred_probs = model.predict(X_test)

# Convert predicted probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print the metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

7766/7766 ━━━━━━━━━━━━━━━━━━━━ 72s 9ms/step
Accuracy: 0.6952
Precision: 0.6976
Recall: 0.6952
F1-score: 0.6953


## Make predictions on new data



In [ ]:

new_comments = [
    "This is a truly amazing comment! I love it.", # Example of non-sarcastic/non-humorous
    "Wow, that was hilarious! I can't stop laughing.", # Example of humor
    "Oh, you're *so* funny. I'm dying of laughter here.", # Example of sarcasm
]


new_comments_features = tfidf_vectorizer.transform(new_comments)

# Make predictions using the trained model
new_predictions_probs = model.predict(new_comments_features)

# Convert predicted probabilities to class labels
# 0: not sarcastic, 1: sarcastic, 2: humor
new_predictions = np.argmax(new_predictions_probs, axis=1)

# Display the predictions
print("Predictions for new comments:")
for comment, prediction in zip(new_comments, new_predictions):
    label = "Not Sarcastic/Humorous" if prediction == 0 else ("Sarcastic" if prediction == 1 else "Humor")
    print(f"Comment: '{comment}' -> Predicted Label: {label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 678ms/step
Predictions for new comments:
Comment: 'This is a truly amazing comment! I love it.' -> Predicted Label: Not Sarcastic/Humorous
Comment: 'Wow, that was hilarious! I can't stop laughing.' -> Predicted Label: Sarcastic
Comment: 'Oh, you're *so* funny. I'm dying of laughter here.' -> Predicted Label: Sarcastic


# Task
Create a simple website using Flask where a user can input text, and the website will use the previously trained TensorFlow model and TF-IDF vectorizer to predict the category of the text. The website should display the prediction result to the user.

## Save the trained model

### Subtask:
Save the trained TensorFlow model and the fitted TF-IDF vectorizer to files so they can be loaded later in the web application.


**Reasoning**:
Save the trained TensorFlow model and the fitted TF-IDF vectorizer to files for later use in the web application.



In [ ]:
import pickle
import os

# Create a directory to save the model and vectorizer if it doesn't exist
save_dir = 'model_and_vectorizer'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Save the trained TensorFlow model
model_save_path = os.path.join(save_dir, 'sarcasm_humor_model.h5') # Using HDF5 format
model.save(model_save_path)
print(f"TensorFlow model saved to: {model_save_path}")

# Save the fitted TF-IDF vectorizer
vectorizer_save_path = os.path.join(save_dir, 'tfidf_vectorizer.pkl')
with open(vectorizer_save_path, 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f"TF-IDF vectorizer saved to: {vectorizer_save_path}")

TensorFlow model saved to: model_and_vectorizer/sarcasm_humor_model.h5
TF-IDF vectorizer saved to: model_and_vectorizer/tfidf_vectorizer.pkl


## Choose a web framework




## Create the web application structure



In [ ]:
import os

# Define the main application file name
app_file = 'app.py'

# Define the templates folder name
templates_folder = 'templates'

# Define the template file names
index_template = os.path.join(templates_folder, 'index.html')
result_template = os.path.join(templates_folder, 'result.html')

# Create the templates folder if it doesn't exist
if not os.path.exists(templates_folder):
    os.makedirs(templates_folder)
    print(f"Created directory: {templates_folder}")

# Create the main application file if it doesn't exist
if not os.path.exists(app_file):
    with open(app_file, 'w') as f:
        f.write("# Flask application goes here\n")
    print(f"Created file: {app_file}")

# Create the index.html template file if it doesn't exist
if not os.path.exists(index_template):
    with open(index_template, 'w') as f:
        f.write("<!-- index.html content goes here -->\n")
    print(f"Created file: {index_template}")

# Create the result.html template file if it doesn't exist
if not os.path.exists(result_template):
    with open(result_template, 'w') as f:
        f.write("<!-- result.html content goes here -->\n")
    print(f"Created file: {result_template}")

print("Basic Flask application structure created.")

Created directory: templates
Created file: app.py
Created file: templates/index.html
Created file: templates/result.html
Basic Flask application structure created.


## Load the model and vectorizer



In [ ]:
from flask import Flask
import tensorflow as tf
import pickle
import os

# Define the paths to the saved model and vectorizer
# Assuming 'model_and_vectorizer' is in the same directory as app.py
model_save_path = os.path.join('model_and_vectorizer', 'sarcasm_humor_model.h5')
vectorizer_save_path = os.path.join('model_and_vectorizer', 'tfidf_vectorizer.pkl')

# Load the trained TensorFlow model
try:
    model = tf.keras.models.load_model(model_save_path)
    print(f"TensorFlow model loaded successfully from: {model_save_path}")
except Exception as e:
    print(f"Error loading TensorFlow model: {e}")
    model = None # Set model to None if loading fails

# Load the fitted TF-IDF vectorizer
try:
    with open(vectorizer_save_path, 'rb') as f:
        tfidf_vectorizer = pickle.load(f)
    print(f"TF-IDF vectorizer loaded successfully from: {vectorizer_save_path}")
except Exception as e:
    print(f"Error loading TF-IDF vectorizer: {e}")
    tfidf_vectorizer = None # Set vectorizer to None if loading fails


# Initialize Flask application
app = Flask(__name__)

# Add a simple route to verify the app is running (optional but helpful)
@app.route('/')
def index():
    return "Flask app is running. Model and vectorizer loading status printed in console."

if __name__ == '__main__':
    # This part is typically not needed in a notebook environment
    # but is included for completeness if running app.py directly
    # print("Attempting to run Flask app...")
    # app.run(debug=True)
    pass # Keep this passive in the notebook context

TensorFlow model loaded successfully from: model_and_vectorizer/sarcasm_humor_model.h5
TF-IDF vectorizer loaded successfully from: model_and_vectorizer/tfidf_vectorizer.pkl


## Create a prediction endpoint


In [ ]:
from flask import request, render_template

# Define the prediction route
@app.route('/predict', methods=['POST'])
def predict():
    user_input = request.form['text']
    prediction_text = "Error: Model or vectorizer not loaded."

    if model is not None and tfidf_vectorizer is not None:
        try:
            # Preprocess the input text
            input_features = tfidf_vectorizer.transform([user_input])

            # Make a prediction
            prediction_probs = model.predict(input_features)
            predicted_label_index = np.argmax(prediction_probs, axis=1)[0]

            # Map the predicted label index to a human-readable category
            label_map = {0: "Not Sarcastic/Humorous", 1: "Sarcastic", 2: "Humor"}
            prediction_text = label_map.get(predicted_label_index, "Unknown Category")

        except Exception as e:
            prediction_text = f"Error during prediction: {e}"
    else:
        prediction_text = "Error: Model or vectorizer could not be loaded on startup."


    return render_template('result.html', original_text=user_input, prediction=prediction_text)


## Design the user interface




In [ ]:
# # Create the content for index.html
# index_html_content = """
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Sarcasm and Humor Detector</title>
#     <style>
#         body { font-family: Arial, sans-serif; margin: 20px; }
#         h1 { color: #333; }
#         form { margin-top: 20px; }
#         textarea { width: 80%; height: 100px; padding: 10px; border: 1px solid #ccc; }
#         button { padding: 10px 15px; background-color: #007bff; color: white; border: none; cursor: pointer; }
#         button:hover { background-color: #0056b3; }
#     </style>
# </head>
# <body>
#     <h1>Enter Text to Classify</h1>
#     <form action="/predict" method="post">
#         <textarea name="text" placeholder="Enter your text here..." required></textarea><br>
#         <button type="submit">Predict</button>
#     </form>
# </body>
# </html>
# """

# # Create the content for result.html
# result_html_content = """
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Prediction Result</title>
#     <style>
#         body { font-family: Arial, sans-serif; margin: 20px; }
#         h1 { color: #333; }
#         p { margin-top: 10px; }
#         strong { color: #007bff; }
#     </style>
# </head>
# <body>
#     <h1>Prediction Result</h1>
#     <p><strong>Original Text:</strong> {{ original_text }}</p>
#     <p><strong>Predicted Category:</strong> {{ prediction }}</p>
#     <p><a href="/">Go Back</a></p>
# </body>
# </html>
# """

# # Write the content to the files
# index_template_path = os.path.join('templates', 'index.html')
# result_template_path = os.path.join('templates', 'result.html')

# with open(index_template_path, 'w') as f:
#     f.write(index_html_content)
# print(f"Created/Updated {index_template_path}")

# with open(result_template_path, 'w') as f:
#     f.write(result_html_content)
# print(f"Created/Updated {result_template_path}")


Created/Updated templates/index.html
Created/Updated templates/result.html


## Run the web application



In [ ]:
# # Append the standard Flask runner code to app.py
# app_file = 'app.py'

# runner_code = """

# if __name__ == '__main__':
#     # Recommended to run this from a terminal using 'python app.py'
#     # Running directly in some notebook environments might require specific extensions
#     # or can block execution.
#     print("Starting Flask development server...")
#     app.run(debug=True)
# """

# with open(app_file, 'a') as f:
#     f.write(runner_code)

# print(f"Added Flask runner code to {app_file}")


Added Flask runner code to app.py


# HANDLING EMOJIS AND ABBREVATIONS


In [1]:
# -*- coding: utf-8 -*-
"""
Final Advanced Sarcasm and Humor Detector
This script trains a sophisticated NLP model using an Embedding layer and an LSTM network
to classify text as sarcastic, humorous, or neutral.
"""

# Step 0: Install necessary libraries
!pip install emoji --quiet

# Step 1: Import all required libraries
import pandas as pd
import numpy as np
import re
import emoji
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("TensorFlow Version:", tf.__version__)

# Step 2: Load the original datasets
try:
    # Make sure to upload these files to your Colab session's file directory
    df_sarcasm = pd.read_csv('/content/drive/MyDrive/NLP prj/sarcasm/train-balanced-sarcasm.csv')
    df_jokes = pd.read_csv('/content/drive/MyDrive/NLP prj/humor/shortjokes.csv')
    print("✅ Datasets loaded successfully!")
except FileNotFoundError:
    print("❌ Error: Make sure 'train-balanced-sarcasm.csv' and 'shortjokes.csv' are uploaded to your Colab session!")
    # Dummy dataframes to prevent script from crashing if files are not found
    df_sarcasm = pd.DataFrame(columns=['comment', 'label'])
    df_jokes = pd.DataFrame(columns=['Joke'])


# Step 3: Define the Advanced Preprocessing Function
# Expanded dictionary for better handling of social media slang
slang_dict = {
    "lol": "laughing out loud",
    "lmfao": "laughing my freaking ass off",
    "smh": "shaking my head",
    "imo": "in my opinion",
    "ikr": "I know right",
    "omg": "oh my god",
    "btw": "by the way",
    "idk": "I don't know",
    "tbh": "to be honest",
    "ftw": "for the win",
    "irl": "in real life"
}

def preprocess_text(text):
    """
    Cleans and prepares text data for NLP models.
    Handles emojis, slang, URLs, mentions, and other noise.
    """
    if not isinstance(text, str):
        return ""
    # Convert emojis to their text description
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Convert to lowercase
    text = text.lower()
    # Replace slang
    words = text.split()
    expanded_words = [slang_dict.get(word, word) for word in words]
    text = " ".join(expanded_words)
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (but keep the text)
    text = re.sub(r'#', '', text)
    # Remove non-alphanumeric characters (optional, can help clean noise)
    text = re.sub(r'[^a-z0-9\s_]', '', text)
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

print("✅ Preprocessing function defined with expanded slang dictionary.")


# Step 4: Prepare and Preprocess the DataFrames
# Drop null values
df_sarcasm.dropna(subset=['comment'], inplace=True)
df_jokes.dropna(subset=['Joke'], inplace=True)

# Apply the preprocessing function
print("\nPreprocessing text data... (This may take a minute)")
df_sarcasm['comment'] = df_sarcasm['comment'].apply(preprocess_text)
df_jokes['Joke'] = df_jokes['Joke'].apply(preprocess_text)
print("✅ Text preprocessing complete.")

# Prepare sarcasm data (label 0 = neutral, label 1 = sarcastic)
df_sarcasm_processed = df_sarcasm[['comment', 'label']].copy()

# Prepare humor data (label 2 = humor)
df_jokes_processed = df_jokes[['Joke']].copy()
df_jokes_processed.rename(columns={'Joke': 'comment'}, inplace=True)
df_jokes_processed['label'] = 2


# Step 5: Combine Data and Convert to Numerical Sequences
# Combine all data into a single DataFrame
combined_df = pd.concat([df_sarcasm_processed, df_jokes_processed], ignore_index=True)

# Separate comments and labels
comments = combined_df['comment']
labels = combined_df['label']

# Initialize and fit the Keras Tokenizer
max_words = 15000  # Size of our vocabulary
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(comments)

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(comments)

# Pad sequences to ensure they are all the same length
maxlen = 100  # Max number of words per comment
X = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
y = labels.values

print("\nShape of data tensor (X):", X.shape)
print("Shape of label tensor (y):", y.shape)


# Step 6: Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\n✅ Data split into training and testing sets.")
print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])


# Step 7: Build the Advanced LSTM Model
print("\nBuilding the LSTM model...")
model = Sequential([
    # Embedding layer: Turns word indices into dense vectors of a fixed size.
    Embedding(input_dim=max_words, output_dim=128, input_length=maxlen),
    # SpatialDropout1D: Regularization to prevent overfitting.
    SpatialDropout1D(0.3),
    # LSTM layer: Processes sequences, capturing contextual information.
    LSTM(64, dropout=0.3, recurrent_dropout=0.3),
    # Dense output layer: For classification.
    Dense(3, activation='softmax')  # 3 classes for neutral, sarcastic, humor
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.5 MB/s eta 0:00:00
TensorFlow Version: 2.19.0
✅ Datasets loaded successfully!
✅ Preprocessing function defined with expanded slang dictionary.

Preprocessing text data... (This may take a minute)
✅ Text preprocessing complete.

Shape of data tensor (X): (1242428, 100)
Shape of label tensor (y): (1242428,)

✅ Data split into training and testing sets.
Training set size: 993942
Testing set size: 248486

Building the LSTM model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/2
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1053s 336ms/step - accuracy: 0.4053 - loss: 1.0469 - val_accuracy: 0.4066 - val_loss: 1.0445
Epoch 2/2
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1038s 334ms/step - accuracy: 0.4057 - loss: 1.0452 - val_accuracy: 0.4078 - val_loss: 1.0444
✅ Model training complete.

Evaluating the model on the test set...
7766/7766 ━━━━━━━━━━━━━━━━━━━━ 473s 61ms/step
Accuracy: 0.4068
Precision: 0.1655
Recall: 0.4068
F1-score: 0.2353

--- Testing with new comments ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Comment: 'Oh great, another meeting that could have been an email 🙄'
-> Predicted: Not Sarcastic/Humorous

Comment: 'Why did the scarecrow win an award? Because he was outstanding in his field!'
-> Predicted: Not Sarcastic/Humorous

Comment: 'I'm so excited for the weekend!'
-> Predicted: Not Sarcastic/Humorous

Comment: 'lmfao I just love it when my code breaks for no reason smh'
-> Predicted: Not Sarcastic/Humorous



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:

# Step 8: Train the Model
print("\nStarting model training...")
history = model.fit(X_train, y_train,
                    epochs=5,  # Increase epochs for better performance, but 2 is good for a quick run
                    batch_size=256,
                    validation_split=0.2,
                    verbose=1)
print("✅ Model training complete.")


# Step 9: Evaluate the Model on the Test Set
print("\nEvaluating the model on the test set...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")


# Step 10: Make Predictions on New, Unseen Data
print("\n--- Testing with new comments ---")
new_comments = [
    "Oh great, another meeting that could have been an email 🙄", # Sarcastic
    "Why did the scarecrow win an award? Because he was outstanding in his field!", # Humor
    "I'm so excited for the weekend!", # Neutral
    "lmfao I just love it when my code breaks for no reason smh" # Sarcastic
]

# The prediction pipeline
def predict_text(text_list):
    # Preprocess the text
    processed_texts = [preprocess_text(t) for t in text_list]
    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(processed_texts)
    # Pad the sequences
    padded_sequences = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
    # Make predictions
    predictions = model.predict(padded_sequences)
    # Get the class with the highest probability
    predicted_classes = np.argmax(predictions, axis=1)
    return predicted_classes

# Run predictions
predictions = predict_text(new_comments)

# Display results
label_map = {0: "Not Sarcastic/Humorous", 1: "Sarcastic", 2: "Humor"}
for i, comment in enumerate(new_comments):
    print(f"Comment: '{comment}'")
    print(f"-> Predicted: {label_map[predictions[i]]}\n")


Starting model training...
Epoch 1/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1053s 339ms/step - accuracy: 0.4058 - loss: 1.0456 - val_accuracy: 0.4078 - val_loss: 1.0443
Epoch 2/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1036s 334ms/step - accuracy: 0.4070 - loss: 1.0449 - val_accuracy: 0.4078 - val_loss: 1.0443
Epoch 3/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1002s 323ms/step - accuracy: 0.4073 - loss: 1.0448 - val_accuracy: 0.4078 - val_loss: 1.0444
Epoch 4/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1052s 326ms/step - accuracy: 0.5127 - loss: 0.9153 - val_accuracy: 0.7036 - val_loss: 0.6301
Epoch 5/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1017s 327ms/step - accuracy: 0.7079 - loss: 0.6197 - val_accuracy: 0.7224 - val_loss: 0.5878
✅ Model training complete.

Evaluating the model on the test set...
7766/7766 ━━━━━━━━━━━━━━━━━━━━ 453s 58ms/step
Accuracy: 0.7243
Precision: 0.7246
Recall: 0.7243
F1-score: 0.7238

--- Testing with new comments ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Comment: 'Oh great, another meeting that could h

# saving the model


In [4]:
import pickle
import os

# Define the directory to save your files
save_dir = 'model_artifacts'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 1. Save the trained Keras model
# We use the recommended '.keras' format
model_path = os.path.join(save_dir, 'sarcasm_humor_model.keras')
model.save(model_path)
print(f"✅ Model saved successfully to: {model_path}")

# 2. Save the fitted Tokenizer
# We use pickle to save the tokenizer object
tokenizer_path = os.path.join(save_dir, 'tokenizer.pkl')
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"✅ Tokenizer saved successfully to: {tokenizer_path}")

✅ Model saved successfully to: model_artifacts/sarcasm_humor_model.keras
✅ Tokenizer saved successfully to: model_artifacts/tokenizer.pkl
